<a href="https://colab.research.google.com/github/JustinRSK/2025_ML_EES/blob/main/Project/Version2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
!pip -q install rasterio scikit-learn numpy matplotlib


In [21]:
import numpy as np
import rasterio
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix


In [22]:
features_path = "/content/Features_2056_FINAL.tif"
labels_path   = "/content/labels_5classes_2056.tif"


In [23]:
import rasterio

with rasterio.open(features_path) as fx:
    print("Features:")
    print(" bands:", fx.count)
    print(" size:", fx.width, fx.height)
    print(" dtype:", fx.dtypes)

with rasterio.open(labels_path) as ly:
    print("Labels:")
    print(" size:", ly.width, ly.height)
    print(" dtype:", ly.dtypes)


Features:
 bands: 7
 size: 4251 3319
 dtype: ('float32', 'float32', 'float32', 'float32', 'float32', 'float32', 'float32')
Labels:
 size: 3253 2540
 dtype: ('uint8',)


In [24]:
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

features_path = "/content/Features_2056_FINAL.tif"
labels_path   = "/content/labels_5classes_2056.tif"

# --- read features reference grid ---
with rasterio.open(features_path) as fx:
    fx_profile   = fx.profile.copy()
    dst_crs      = fx.crs
    dst_transform= fx.transform
    dst_h        = fx.height
    dst_w        = fx.width

# --- read labels ---
with rasterio.open(labels_path) as ly:
    y_src        = ly.read(1)
    src_crs      = ly.crs
    src_transform= ly.transform
    src_nodata   = ly.nodata if ly.nodata is not None else 0

# --- align labels to features grid (nearest neighbor keeps class integers) ---
y = np.zeros((dst_h, dst_w), dtype=np.uint8)

reproject(
    source=y_src,
    destination=y,
    src_transform=src_transform,
    src_crs=src_crs,
    dst_transform=dst_transform,
    dst_crs=dst_crs,
    src_nodata=src_nodata,
    dst_nodata=0,
    resampling=Resampling.nearest
)

print("Aligned labels shape:", y.shape)
print("Unique label values:", np.unique(y))


Aligned labels shape: (3319, 4251)
Unique label values: [0 1 2 3 4 5]


In [25]:
with rasterio.open(features_path) as fx:
    X = fx.read()  # (bands, rows, cols)

print("Features shape:", X.shape, "dtype:", X.dtype)
print("Now match?", (X.shape[1], X.shape[2]) == y.shape)


Features shape: (7, 3319, 4251) dtype: float32
Now match? True


Build training table (pixels) + train/test split

In [26]:
import numpy as np
from sklearn.model_selection import train_test_split

# X: (bands, rows, cols)  y: (rows, cols) already in memory
bands, rows, cols = X.shape

# Flatten to per-pixel table
Xpix = np.moveaxis(X, 0, -1).reshape(-1, bands)   # (N, bands)
ypix = y.reshape(-1)                              # (N,)

# keep only labeled pixels (ignore 0 = background)
mask = (ypix != 0) & np.all(np.isfinite(Xpix), axis=1)
X_train_all = Xpix[mask]
y_train_all = ypix[mask]

print("Training pixels:", X_train_all.shape, "Classes:", np.unique(y_train_all))

# split
Xtr, Xte, ytr, yte = train_test_split(
    X_train_all, y_train_all,
    test_size=0.2,
    random_state=42,
    stratify=y_train_all
)

print("Train:", Xtr.shape, "Test:", Xte.shape)


Training pixels: (5644, 7) Classes: [1 2 3 4 5]
Train: (4515, 7) Test: (1129, 7)


Train Random Forest + evaluation

In [27]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

rf = RandomForestClassifier(
    n_estimators=400,
    max_features="sqrt",
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42
)

rf.fit(Xtr, ytr)

pred = rf.predict(Xte)
print("Confusion matrix:\n", confusion_matrix(yte, pred))
print("\nReport:\n", classification_report(yte, pred, digits=3))


Confusion matrix:
 [[ 42   2   1   4   0]
 [  0 127   2   3   0]
 [  0   1  50  28   0]
 [  1   0  11 239   0]
 [  0   0   0   0 618]]

Report:
               precision    recall  f1-score   support

           1      0.977     0.857     0.913        49
           2      0.977     0.962     0.969       132
           3      0.781     0.633     0.699        79
           4      0.872     0.952     0.910       251
           5      1.000     1.000     1.000       618

    accuracy                          0.953      1129
   macro avg      0.921     0.881     0.898      1129
weighted avg      0.953     0.953     0.952      1129



Predict the whole AOI raster + confidence

In [30]:
proba = rf.predict_proba(Xpix)
class_ids = rf.classes_

pred_flat = class_ids[np.argmax(proba, axis=1)].astype(np.uint8)
conf_flat = np.max(proba, axis=1).astype(np.float32)

pred_map = pred_flat.reshape(rows, cols)
conf_map = conf_flat.reshape(rows, cols)


In [32]:
profile_cls = base_profile.copy()
profile_cls.update(
    dtype=rasterio.uint8,
    count=1,
    nodata=0,
    compress="lzw"
)

profile_conf = base_profile.copy()
profile_conf.update(
    dtype=rasterio.float32,
    count=1,
    nodata=-9999,
    compress="lzw"
)

with rasterio.open("/content/prediction_classes.tif", "w", **profile_cls) as dst:
    dst.write(pred_map.astype(np.uint8), 1)

with rasterio.open("/content/prediction_confidence.tif", "w", **profile_conf) as dst:
    dst.write(conf_map.astype(np.float32), 1)
